# GBot Memory — Embedding Model Benchmark

Türkçe memory fact'ler için en iyi embedding modelini seçmek.

**Kriterler:**
- Aynı konudaki çelişkili fact'ler yüksek similarity (>0.70) → gri bölge → LLM karar
- Alakasız fact'ler düşük similarity (<0.70) → otomatik ADD
- Gap ne kadar büyükse model o kadar iyi ayırt ediyor

In [ ]:
import httpx
import math
import os
import json
from dotenv import load_dotenv

load_dotenv('/root/gbot/.env')
API_KEY = os.environ['OPENROUTER_API_KEY']
print(f'Key: {API_KEY[:15]}...')

In [ ]:
def get_embeddings(model: str, texts: list[str]) -> list[list[float]]:
    """OpenRouter embedding API call."""
    r = httpx.post(
        'https://openrouter.ai/api/v1/embeddings',
        headers={'Authorization': f'Bearer {API_KEY}'},
        json={'model': model, 'input': texts},
        timeout=30
    )
    data = r.json()
    if 'data' not in data:
        raise ValueError(f"API error: {data.get('error', {}).get('message', '?')}")
    return [d['embedding'] for d in data['data']], data.get('usage', {})


def cosine(a: list[float], b: list[float]) -> float:
    """Cosine similarity between two vectors."""
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(x * x for x in b))
    return dot / (na * nb) if na and nb else 0.0

## Test Cümleleri

Çelişki grupları + alakasız fact'ler. İyi bir model:
- Aynı gruptaki çelişkileri yüksek similarity ile eşler
- Farklı gruptakileri düşük similarity ile ayırır

In [ ]:
# Çelişki grupları — aynı konu, farklı bilgi
CONFLICT_PAIRS = [
    # Konum
    ('Ömer İstanbul\'da yaşıyor', 'Ömer Ankara\'ya taşındı'),
    # İş
    ('Ömer HangiKredi\'de çalışıyor', 'Ömer Trendyol\'a geçti'),
    # Tercih
    ('Kahve sevmiyor çay tercih ediyor', 'Son zamanlarda kahveye başladı'),
    # Diyet
    ('Ömer vejetaryen', 'Ömer artık et yiyor'),
    # Teknoloji
    ('Python ve Go kullanıyor', 'Rust\'a geçiş yaptı artık Go kullanmıyor'),
    # Medeni hal
    ('Ömer bekar', 'Ömer evlendi'),
    # Araç
    ('Toyota Corolla kullanıyor', 'Arabasını sattı artık araba kullanmıyor'),
]

# Alakasız çiftler — farklı konular
UNRELATED_PAIRS = [
    ('Ömer İstanbul\'da yaşıyor', 'Ömer vejetaryen'),
    ('Ömer İstanbul\'da yaşıyor', 'Python ve Go kullanıyor'),
    ('Kahve sevmiyor çay tercih ediyor', 'HangiKredi\'de çalışıyor'),
    ('Ömer bekar', 'Sabahları borsa takip ediyor'),
    ('Toyota Corolla kullanıyor', 'Ömer vejetaryen'),
    ('Ömer evlendi', 'Rust\'a geçiş yaptı'),
]

# Duplicate çiftler — aynı bilgi farklı ifade
DUPLICATE_PAIRS = [
    ('Ömer İstanbul\'da yaşıyor', 'İstanbul\'da ikamet ediyor'),
    ('Kahve sevmiyor', 'Kahveden hoşlanmıyor'),
    ('Python kullanıyor', 'Python ile geliştirme yapıyor'),
    ('Ömer vejetaryen', 'Et yemiyor vejetaryen beslenme tercih ediyor'),
    ('HangiKredi\'de çalışıyor', 'HangiKredi şirketinde işe gidiyor'),
]

# Tüm unique cümleler
all_texts = list(set(
    [t for pair in CONFLICT_PAIRS for t in pair] +
    [t for pair in UNRELATED_PAIRS for t in pair] +
    [t for pair in DUPLICATE_PAIRS for t in pair]
))
print(f'Toplam {len(all_texts)} unique cümle')

In [ ]:
def benchmark_model(model: str, all_texts: list[str]):
    """Bir modeli tüm test çiftleriyle değerlendir."""
    try:
        embs, usage = get_embeddings(model, all_texts)
    except Exception as e:
        return None, str(e)
    
    text_to_emb = dict(zip(all_texts, embs))
    dim = len(embs[0])
    
    def pair_sim(a, b):
        return cosine(text_to_emb[a], text_to_emb[b])
    
    # Conflict similarities (should be HIGH — same topic, different info)
    conflict_sims = [pair_sim(a, b) for a, b in CONFLICT_PAIRS]
    
    # Unrelated similarities (should be LOW — different topics)
    unrelated_sims = [pair_sim(a, b) for a, b in UNRELATED_PAIRS]
    
    # Duplicate similarities (should be VERY HIGH — same info)
    duplicate_sims = [pair_sim(a, b) for a, b in DUPLICATE_PAIRS]
    
    avg_conflict = sum(conflict_sims) / len(conflict_sims)
    avg_unrelated = sum(unrelated_sims) / len(unrelated_sims)
    avg_duplicate = sum(duplicate_sims) / len(duplicate_sims)
    gap = avg_conflict - avg_unrelated
    
    return {
        'model': model,
        'dim': dim,
        'avg_conflict': avg_conflict,
        'avg_unrelated': avg_unrelated,
        'avg_duplicate': avg_duplicate,
        'gap': gap,
        'conflict_sims': conflict_sims,
        'unrelated_sims': unrelated_sims,
        'duplicate_sims': duplicate_sims,
        'usage': usage,
    }, None

## Tüm Modelleri Test Et

In [ ]:
MODELS = [
    # OpenAI
    'openai/text-embedding-3-small',
    'openai/text-embedding-3-large',
    'openai/text-embedding-ada-002',
    # Sentence Transformers
    'sentence-transformers/all-mpnet-base-v2',
    'sentence-transformers/all-minilm-l6-v2',
    'sentence-transformers/all-minilm-l12-v2',
    'sentence-transformers/paraphrase-minilm-l6-v2',
    'sentence-transformers/multi-qa-mpnet-base-dot-v1',
    # BAAI
    'baai/bge-m3',
    'baai/bge-large-en-v1.5',
    'baai/bge-base-en-v1.5',
    # Others
    'intfloat/multilingual-e5-large',
    'intfloat/e5-large-v2',
    'thenlper/gte-large',
    'qwen/qwen3-embedding-8b',
    'qwen/qwen3-embedding-4b',
    'google/gemini-embedding-001',
    'mistralai/mistral-embed-2312',
    'mistralai/codestral-embed-2505',
    'perplexity/pplx-embed-v1-0.6b',
    'nvidia/llama-nemotron-embed-vl-1b-v2:free',
]

results = []
for model in MODELS:
    print(f'Testing {model}...', end=' ')
    result, error = benchmark_model(model, all_texts)
    if error:
        print(f'FAIL: {error[:60]}')
    else:
        results.append(result)
        mark = 'GREAT' if result['gap'] > 0.25 else ('OK' if result['gap'] > 0.15 else 'BAD')
        print(f"gap={result['gap']:+.3f} conflict={result['avg_conflict']:.3f} unrelated={result['avg_unrelated']:.3f} dup={result['avg_duplicate']:.3f} [{mark}]")

print(f'\nTested {len(results)}/{len(MODELS)} models')

## Sıralama — En İyi Modeller

In [ ]:
# Gap'e göre sırala (büyük = daha iyi ayrım)
results.sort(key=lambda r: r['gap'], reverse=True)

print(f'{"Model":55s} {"Dim":>5s} {"Conflict":>8s} {"Unrelated":>9s} {"Duplicate":>9s} {"Gap":>6s} {"Grade":>6s}')
print('-' * 100)
for r in results:
    mark = 'GREAT' if r['gap'] > 0.25 else ('OK' if r['gap'] > 0.15 else 'BAD')
    print(f"{r['model']:55s} {r['dim']:5d} {r['avg_conflict']:8.3f} {r['avg_unrelated']:9.3f} {r['avg_duplicate']:9.3f} {r['gap']:+6.3f} {mark:>6s}")

## Detaylı Analiz — Top 3 Model

In [ ]:
for r in results[:3]:
    print(f"\n{'='*60}")
    print(f"Model: {r['model']} (dim={r['dim']})")
    print(f"Gap: {r['gap']:+.3f}  |  Avg conflict: {r['avg_conflict']:.3f}  |  Avg unrelated: {r['avg_unrelated']:.3f}  |  Avg duplicate: {r['avg_duplicate']:.3f}")
    
    print(f"\n  Conflict pairs (should be HIGH, >0.70):")
    for (a, b), sim in zip(CONFLICT_PAIRS, r['conflict_sims']):
        zone = 'NOOP' if sim > 0.95 else ('GREY' if sim > 0.70 else 'ADD!')
        print(f"    {sim:.3f} [{zone:4s}]  {a[:30]:30s} <> {b[:30]}")
    
    print(f"\n  Unrelated pairs (should be LOW, <0.70):")
    for (a, b), sim in zip(UNRELATED_PAIRS, r['unrelated_sims']):
        zone = 'NOOP' if sim > 0.95 else ('GREY!' if sim > 0.70 else 'ADD')
        print(f"    {sim:.3f} [{zone:5s}]  {a[:30]:30s} <> {b[:30]}")
    
    print(f"\n  Duplicate pairs (should be VERY HIGH, >0.90):")
    for (a, b), sim in zip(DUPLICATE_PAIRS, r['duplicate_sims']):
        ok = 'OK' if sim > 0.90 else ('WEAK' if sim > 0.80 else 'FAIL')
        print(f"    {sim:.3f} [{ok:4s}]  {a[:30]:30s} <> {b[:30]}")

## Cascading Threshold Analizi

En iyi model için optimal threshold'ları bul:
- `noop_threshold`: üstü = duplicate (SKIP)
- `add_threshold`: altı = yeni bilgi (ADD)
- Arada = gri bölge → LLM karar

In [ ]:
if results:
    best = results[0]
    print(f"Best model: {best['model']}\n")
    
    all_sims = {
        'conflict': best['conflict_sims'],
        'unrelated': best['unrelated_sims'],
        'duplicate': best['duplicate_sims'],
    }
    
    # Find optimal thresholds
    min_dup = min(best['duplicate_sims'])
    max_conflict = max(best['conflict_sims'])
    max_unrelated = max(best['unrelated_sims'])
    min_conflict = min(best['conflict_sims'])
    
    print(f"Duplicate range:  {min(best['duplicate_sims']):.3f} — {max(best['duplicate_sims']):.3f}")
    print(f"Conflict range:   {min_conflict:.3f} — {max_conflict:.3f}")
    print(f"Unrelated range:  {min(best['unrelated_sims']):.3f} — {max_unrelated:.3f}")
    
    suggested_noop = round(min_dup - 0.02, 2)  # just below lowest duplicate
    suggested_add = round(max_unrelated + 0.05, 2)  # just above highest unrelated
    
    print(f"\nSuggested thresholds:")
    print(f"  noop_threshold: {suggested_noop} (> this = duplicate, skip)")
    print(f"  add_threshold:  {suggested_add} (< this = new info, add)")
    print(f"  grey zone:      {suggested_add} — {suggested_noop}")
    
    # Verify with all pairs
    print(f"\nVerification:")
    for label, sims, pairs in [
        ('conflict', best['conflict_sims'], CONFLICT_PAIRS),
        ('unrelated', best['unrelated_sims'], UNRELATED_PAIRS),
        ('duplicate', best['duplicate_sims'], DUPLICATE_PAIRS),
    ]:
        for sim, (a, b) in zip(sims, pairs):
            if sim > suggested_noop:
                decision = 'NOOP'
            elif sim < suggested_add:
                decision = 'ADD'
            else:
                decision = 'GREY→LLM'
            correct = (label == 'duplicate' and decision == 'NOOP') or \
                      (label == 'unrelated' and decision == 'ADD') or \
                      (label == 'conflict' and decision == 'GREY→LLM')
            mark = '✓' if correct else '✗'
            print(f"  {mark} [{label:9s}] {sim:.3f} → {decision:9s}  {a[:25]} <> {b[:25]}")